In [ ]:
!pip install statsmodels openpyxl

# Inter-Rater Agreement

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
from sklearn.metrics import cohen_kappa_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Interpretasi Kappa
def interpret_kappa(k):
    if k < 0:
        return "Poor"
    elif k < 0.20:
        return "Slight"
    elif k < 0.40:
        return "Fair"
    elif k < 0.60:
        return "Moderate"
    elif k < 0.80:
        return "Substantial"
    else:
        return "Almost Perfect"

In [ ]:
def calculate_iaa(file1, file2, file3, label="IAA Result"):
    """
    Menghitung Fleiss' Kappa dan Cohen's Kappa
    dari 3 file anotator.
    """

    # Load file
    df1 = pd.read_excel(file1)[["tweet_url", "full_text", "label"]].rename(columns={"label": "A1"})
    df2 = pd.read_excel(file2)[["tweet_url", "full_text", "label"]].rename(columns={"label": "A2"})
    df3 = pd.read_excel(file3)[["tweet_url", "full_text", "label"]].rename(columns={"label": "A3"})

    # Hapus duplikat tweet
    for df in [df1, df2, df3]:
        df.drop_duplicates(subset="full_text", keep="first", inplace=True)

    # Merge berdasarkan full_text
    merged = (
        df1.merge(df2[["full_text", "A2"]], on="full_text")
           .merge(df3[["full_text", "A3"]], on="full_text")
    )

    # Hapus data kosong
    merged.dropna(subset=["A1", "A2", "A3"], inplace=True)

    # Validasi
    if merged.empty:
        raise ValueError("Tidak ada data yang bisa dihitung setelah proses merge.")

    # Fleiss' Kappa
    labels_array = merged[["A1", "A2", "A3"]].values

    # Ubah ke format matriks (n_items × n_categories)
    agg_data, _ = aggregate_raters(labels_array)

    # Hitung Fleiss' Kappa
    fk = fleiss_kappa(agg_data, method="fleiss")

    # Cohen's Kappa Pairwise
    ck_12 = cohen_kappa_score(merged["A1"], merged["A2"])
    ck_13 = cohen_kappa_score(merged["A1"], merged["A3"])
    ck_23 = cohen_kappa_score(merged["A2"], merged["A3"])

    avg_ck = np.mean([ck_12, ck_13, ck_23])

    print(f"{label} ({len(merged)} data)")
    print(f"Fleiss' Kappa     : {fk:.4f} → {interpret_kappa(fk)}")
    print(f"Cohen A1 & A2     : {ck_12:.4f} → {interpret_kappa(ck_12)}")
    print(f"Cohen A1 & A3     : {ck_13:.4f} → {interpret_kappa(ck_13)}")
    print(f"Cohen A2 & A3     : {ck_23:.4f} → {interpret_kappa(ck_23)}")
    print(f"Rata-rata Cohen κ : {avg_ck:.4f} → {interpret_kappa(avg_ck)}")

    hasil = {
        "fleiss": fk,
        "ck_12": ck_12,
        "ck_13": ck_13,
        "ck_23": ck_23,
        "avg_cohen": avg_ck
    }

    return hasil, merged

In [ ]:
def tweet_calibration(merged, output_file="kalibrasi.xlsx"):
    """
    Identifikasi tweet bermasalah dan simpan ke Excel.
    """

    merged = merged.copy()

    merged["A3_outlier"] = (
        (merged["A3"] != merged["A1"]) &
        (merged["A3"] != merged["A2"])
    )

    merged["A1A2_sepakat"] = (
        merged["A1"] == merged["A2"]
    )

    merged["semua_beda"] = (
        (merged["A1"] != merged["A2"]) &
        (merged["A1"] != merged["A3"]) &
        (merged["A2"] != merged["A3"])
    )

    # Prioritas
    p1 = merged[merged["A3_outlier"] & merged["A1A2_sepakat"]]
    p2 = merged[merged["semua_beda"]]

    p3 = merged[
        ~merged["A3_outlier"] &
        ~merged["semua_beda"] &
        (merged[["A1", "A2", "A3"]].nunique(axis=1) > 1)
    ]

    print("\nHASIL KALIBRASI")
    print(f"Prioritas 1 (A3 outlier) : {len(p1)} data")
    print(f"Prioritas 2 (Semua beda) : {len(p2)} data")
    print(f"Konflik parsial          : {len(p3)} data")

    cols = ["tweet_url", "full_text", "A1", "A2", "A3"]

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        p1[cols].to_excel(writer, sheet_name="P1_A3_Outlier", index=False)
        p2[cols].to_excel(writer, sheet_name="P2_Semua_Beda", index=False)
        p3[cols].to_excel(writer, sheet_name="P3_Konflik_Parsial", index=False)

    print(f"File kalibrasi disimpan ke: {output_file}")

In [ ]:
def summary(hasil):
    return pd.DataFrame([
        {
            "Metrik": "Fleiss' Kappa",
            "Nilai": round(hasil["fleiss"], 4),
            "Interpretasi": interpret_kappa(hasil["fleiss"])
        },
        {
            "Metrik": "Cohen A1 & A2",
            "Nilai": round(hasil["ck_12"], 4),
            "Interpretasi": interpret_kappa(hasil["ck_12"])
        },
        {
            "Metrik": "Cohen A1 & A3",
            "Nilai": round(hasil["ck_13"], 4),
            "Interpretasi": interpret_kappa(hasil["ck_13"])
        },
        {
            "Metrik": "Cohen A2 & A3",
            "Nilai": round(hasil["ck_23"], 4),
            "Interpretasi": interpret_kappa(hasil["ck_23"])
        },
        {
            "Metrik": "Rata-rata Cohen κ",
            "Nilai": round(hasil["avg_cohen"], 4),
            "Interpretasi": interpret_kappa(hasil["avg_cohen"])
        }
    ])

## IAA_1

In [ ]:
# Memuat File
a1 = '/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_100_150_1.xlsx'
a2 = '/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_100_150_2.xlsx'
a3 = '/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_100_150_3.xlsx'

In [ ]:
hasil_iaa1, merged_iaa1 = calculate_iaa(
    a1,
    a2,
    a3,
    label="IAA Batch 1"
)
summary_iaa1 = summary(hasil_iaa1)

IAA Batch 1 (87 data)
Fleiss' Kappa     : 0.2635 → Fair
Cohen A1 & A2     : 0.5781 → Moderate
Cohen A1 & A3     : 0.1158 → Slight
Cohen A2 & A3     : 0.1544 → Slight
Rata-rata Cohen κ : 0.2828 → Fair


In [ ]:
cols = ["tweet_url", "full_text", "A1", "A2", "A3"]

with pd.ExcelWriter("hasil_iaa1.xlsx", engine="openpyxl") as writer:
    summary_iaa1.to_excel(writer, sheet_name="Summary", index=False)
    merged_iaa1[cols].to_excel(writer, sheet_name="Detail", index=False)

print("File disimpan ke: hasil_iaa1.xlsx")

File disimpan ke: hasil_iaa1.xlsx


In [ ]:
tweet_calibration(
    merged_iaa1,
    output_file="kalibrasi_iaa1.xlsx"
)


HASIL KALIBRASI
Prioritas 1 (A3 outlier) : 37 data
Prioritas 2 (Semua beda) : 12 data
Konflik parsial          : 16 data
File kalibrasi disimpan ke: kalibrasi_iaa1.xlsx


## IAA_2

In [ ]:
# Memuat File
A1 = '/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_2_1.xlsx'
A2 = '/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_2_2.xlsx'
A3 = '/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_2_3.xlsx'

In [ ]:
hasil_iaa2, merged_iaa2 = calculate_iaa(
    A1,
    A2,
    A3,
    label="IAA Batch 2"
)
summary_iaa2 = summary(hasil_iaa2)

IAA Batch 2 (114 data)
Fleiss' Kappa     : 0.4288 → Moderate
Cohen A1 & A2     : 0.6129 → Substantial
Cohen A1 & A3     : 0.1863 → Slight
Cohen A2 & A3     : 0.5056 → Moderate
Rata-rata Cohen κ : 0.4349 → Moderate


In [ ]:
cols = ["tweet_url", "full_text", "A1", "A2", "A3"]

with pd.ExcelWriter("hasil_iaa2.xlsx", engine="openpyxl") as writer:
    summary_iaa2.to_excel(writer, sheet_name="Summary", index=False)
    merged_iaa2[cols].to_excel(writer, sheet_name="Detail", index=False)

print("File disimpan ke: hasil_iaa2.xlsx")

File disimpan ke: hasil_iaa2.xlsx


In [ ]:
# Tabel perbandingan IAA
metrics = ["fleiss", "ck_12", "ck_13", "ck_23", "avg_cohen"]

labels = [
    "Fleiss' Kappa",
    "Cohen A1 & A2",
    "Cohen A1 & A3",
    "Cohen A2 & A3",
    "Rata-rata Cohen κ"
]

perbandingan_df = pd.DataFrame([
    {
        "Metrik": label,

        "IAA 1": round(hasil_iaa1[metric], 4),
        "Interpretasi IAA 1": interpret_kappa(hasil_iaa1[metric]),

        "IAA 2": round(hasil_iaa2[metric], 4),
        "Interpretasi IAA 2": interpret_kappa(hasil_iaa2[metric]),

        "Selisih (IAA2 - IAA1)": round(
            hasil_iaa2[metric] - hasil_iaa1[metric],
            4
        )
    }

    for metric, label in zip(metrics, labels)
])

perbandingan_df

,Metrik,IAA 1,Interpretasi IAA 1,IAA 2,Interpretasi IAA 2,Selisih (IAA2 - IAA1)
0,Fleiss' Kappa,0.2635,Fair,0.4288,Moderate,0.1653
1,Cohen A1 & A2,0.5781,Moderate,0.6129,Substantial,0.0348
2,Cohen A1 & A3,0.1158,Slight,0.1863,Slight,0.0705
3,Cohen A2 & A3,0.1544,Slight,0.5056,Moderate,0.3512
4,Rata-rata Cohen κ,0.2828,Fair,0.4349,Moderate,0.1522


In [ ]:
with pd.ExcelWriter("perbandingan_iaa.xlsx", engine="openpyxl") as writer:
    perbandingan_df.to_excel(writer, sheet_name="Summary", index=False)

print("File disimpan ke: perbandingan_iaa.xlsx")

File disimpan ke: perbandingan_iaa.xlsx


In [ ]:
tweet_calibration(
    merged_iaa2,
    output_file="kalibrasi_iaa2.xlsx"
)


HASIL KALIBRASI
Prioritas 1 (A3 outlier) : 40 data
Prioritas 2 (Semua beda) : 5 data
Konflik parsial          : 30 data
File kalibrasi disimpan ke: kalibrasi_iaa2.xlsx


## IAA_3

In [ ]:
# Memuat File
anotator1 = '/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_3_1.xlsx'
anotator2 = '/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_3_2.xlsx'
anotator3 = '/content/drive/MyDrive/skripsi/svm_indobert/data/inter_annotator/sample_3_3.xlsx'

In [ ]:
hasil_iaa3, merged_iaa3 = calculate_iaa(
    anotator1,
    anotator2,
    anotator3,
    label="IAA Batch 3"
)
summary_iaa2 = summary(hasil_iaa3)

IAA Batch 3 (90 data)
Fleiss' Kappa     : 0.8897 → Almost Perfect
Cohen A1 & A2     : 0.8649 → Almost Perfect
Cohen A1 & A3     : 0.9547 → Almost Perfect
Cohen A2 & A3     : 0.8501 → Almost Perfect
Rata-rata Cohen κ : 0.8899 → Almost Perfect


In [ ]:
cols = ["tweet_url", "full_text", "A1", "A2", "A3"]

with pd.ExcelWriter("hasil_iaa3.xlsx", engine="openpyxl") as writer:
    summary_iaa2.to_excel(writer, sheet_name="Summary", index=False)
    merged_iaa2[cols].to_excel(writer, sheet_name="Detail", index=False)

print("File disimpan ke: hasil_iaa3.xlsx")

File disimpan ke: hasil_iaa3.xlsx


In [ ]:
# Tabel perbandingan IAA
metrics = ["fleiss", "ck_12", "ck_13", "ck_23", "avg_cohen"]

labels = [
    "Fleiss' Kappa",
    "Cohen A1 & A2",
    "Cohen A1 & A3",
    "Cohen A2 & A3",
    "Rata-rata Cohen κ"
]

perbandingan_df = pd.DataFrame([
    {
        "Metrik": label,

        "IAA 2": round(hasil_iaa2[metric], 4),
        "Interpretasi IAA 2": interpret_kappa(hasil_iaa2[metric]),

        "IAA 3": round(hasil_iaa3[metric], 4),
        "Interpretasi IAA 3": interpret_kappa(hasil_iaa3[metric]),

        "Selisih (IAA3 - IAA2)": round(
            hasil_iaa3[metric] - hasil_iaa2[metric],
            4
        )
    }

    for metric, label in zip(metrics, labels)
])

perbandingan_df

,Metrik,IAA 2,Interpretasi IAA 2,IAA 3,Interpretasi IAA 3,Selisih (IAA3 - IAA2)
0,Fleiss' Kappa,0.4288,Moderate,0.8897,Almost Perfect,0.4609
1,Cohen A1 & A2,0.6129,Substantial,0.8649,Almost Perfect,0.2520
2,Cohen A1 & A3,0.1863,Slight,0.9547,Almost Perfect,0.7684
3,Cohen A2 & A3,0.5056,Moderate,0.8501,Almost Perfect,0.3445
4,Rata-rata Cohen κ,0.4349,Moderate,0.8899,Almost Perfect,0.4550


In [ ]:
with pd.ExcelWriter("perbandingan_iaa2.xlsx", engine="openpyxl") as writer:
    perbandingan_df.to_excel(writer, sheet_name="Summary", index=False)

print("File disimpan ke: perbandingan_iaa2.xlsx")

File disimpan ke: perbandingan_iaa2.xlsx


In [ ]:
tweet_calibration(
    merged_iaa2,
    output_file="kalibrasi_iaa3.xlsx"
)


HASIL KALIBRASI
Prioritas 1 (A3 outlier) : 40 data
Prioritas 2 (Semua beda) : 5 data
Konflik parsial          : 30 data
File kalibrasi disimpan ke: kalibrasi_iaa3.xlsx
